### Random Forest Regression
[RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

In [31]:
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import BaggingRegressor, ExtraTreesRegressor

from sklearn.metrics import r2_score





In [3]:
dataset = Path("datasets")
concrete_data = pd.read_csv(dataset / "concrete_data.csv")

In [4]:
X = concrete_data.drop(columns=["csMPa"])
y = concrete_data["csMPa"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [8]:
from sklearn.ensemble import RandomForestRegressor

rnd_reg = RandomForestRegressor(
    n_estimators=600,
    max_leaf_nodes=12,
    random_state=42,
    n_jobs=7,
    min_samples_leaf=1,
    max_features=1.0
)

rnd_reg.fit(X_train, y_train)
y_pred = rnd_reg.predict(X_test)

r2_score(y_test, y_pred)


0.7069618963223071

In [13]:
important_features = (
    pd.DataFrame(data={
            "feature":X.columns,
            "importance": rnd_reg.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)
)


In [14]:
important_features

,feature,importance
0,age,0.388671
1,cement,0.370082
2,water,0.124968
3,slag,0.056795
4,superplasticizer,0.038590
5,flyash,0.010033
6,fineaggregate,0.007724
7,coarseaggregate,0.003138


BaggingRegressor + DecisionTreeRegressor(with splitter="random") = RandomForestRegressor

In [17]:
base_tree = DecisionTreeRegressor(
    splitter="random",
    max_leaf_nodes=12,
    random_state=42,

)
bag_reg = BaggingRegressor(
    estimator=base_tree,
    n_estimators=600,
    bootstrap=True,
    n_jobs=-1,
    max_samples=1.0,
    random_state=42
)

bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)

r2_score(y_test, y_pred)

0.7016524988362387

In [32]:
importances = np.mean([tree.feature_importances_ for tree in bag_reg.estimators_], axis=0)

In [35]:
important_features_bagging = (
    pd.DataFrame(data={
            "feature":X.columns,
            "importance": importances
}).sort_values(by="importance", ascending=False).reset_index(drop=True)
)
important_features_bagging

,feature,importance
0,age,0.358873
1,cement,0.323365
2,water,0.097615
3,superplasticizer,0.086113
4,slag,0.066314
5,flyash,0.041394
6,fineaggregate,0.017917
7,coarseaggregate,0.008409


ExtraTreesRegressor
[Link](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesRegressor.html)


In [36]:
from sklearn.ensemble import ExtraTreesRegressor

extra_reg = ExtraTreesRegressor(
    n_estimators=600,
    max_leaf_nodes=12,
    n_jobs=-1,
    random_state=42,
)
extra_reg.fit(X_train, y_train)
y_pred = extra_reg.predict(X_test)

r2_score(y_test, y_pred)



0.7016079158939398

In [37]:
important_features_bagging = (
    pd.DataFrame(data={
            "feature":X.columns,
            "importance": extra_reg.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)
)

In [38]:
important_features

,feature,importance
0,age,0.388671
1,cement,0.370082
2,water,0.124968
3,slag,0.056795
4,superplasticizer,0.038590
5,flyash,0.010033
6,fineaggregate,0.007724
7,coarseaggregate,0.003138
